In [54]:
# Import the SageMaker SDK.
import sagemaker
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

from sagemaker.core import image_uris
from sagemaker.train.configs import InputData
from sagemaker.train import ModelTrainer
from sagemaker.core.helper.session_helper import (
    Session,
    get_execution_role
)

from sagemaker.train.configs import (
    CheckpointConfig,
    Compute,
    SourceCode,
    StoppingCondition
)

# Standard library imports.
from importlib.metadata import version
from pathlib import Path


# Display the current notebook directory.
current_directory = Path.cwd()

# Get the execution role.
import boto3


# Create the underlying AWS SDK session.
boto_session = boto3.Session()

# Create the SageMaker SDK V3 session.
sagemaker_session = Session(
    boto_session=boto_session
)

# Read the current AWS Region.
aws_region = sagemaker_session.boto_region_name

# Get the SageMaker execution role.

execution_role = get_execution_role() 
print("Execution role:", execution_role)
sagemaker_session = boto3.Session()

# Display the notebook environment.
print("Current directory:", current_directory)
print("AWS Region:", aws_region)
print("Execution role:", execution_role)
print("SageMaker Core version:", version("sagemaker-core"))
print("SageMaker Train version:", version("sagemaker-train"))
print("SageMaker V3 imports: OK")



Execution role: arn:aws:iam::206440129399:role/service-role/AmazonSageMaker-ExecutionRole-20260730T162226
Current directory: /home/sagemaker-user/ClassMushroomAWS/training
AWS Region: us-east-1
Execution role: arn:aws:iam::206440129399:role/service-role/AmazonSageMaker-ExecutionRole-20260730T162226
SageMaker Core version: 2.18.0
SageMaker Train version: 1.18.0
SageMaker V3 imports: OK


In [55]:
# Define the local project directories.
project_directory = Path.cwd()

training_script = project_directory / "train.py"
requirements_file = project_directory / "requirements.txt"

print("Project directory :", project_directory)
print("Training script   :", training_script)
print("Requirements file :", requirements_file)

print()
print("train.py exists      :", training_script.exists())
print("requirements exists  :", requirements_file.exists())

Project directory : /home/sagemaker-user/ClassMushroomAWS/training
Training script   : /home/sagemaker-user/ClassMushroomAWS/training/train.py
Requirements file : /home/sagemaker-user/ClassMushroomAWS/training/requirements.txt

train.py exists      : True
requirements exists  : True


In [56]:
# Define the S3 input datasets.
train_s3_uri = (
    "s3://mushroom-ml-mikael-2026-000/"
    "processed/parquet/"
    "train_balanced_augmented.parquet"
)

validation_s3_uri = (
    "s3://mushroom-ml-mikael-2026-000/"
    "processed/parquet/"
    "validation.parquet"
)

In [59]:


# Define the remote training configuration.
instance_type = "ml.t3.large"

# Use one epoch for the first pipeline test.
epochs = 1
batch_size = 32
learning_rate = 0.001


# Retrieve the official AWS PyTorch training image.
training_image = image_uris.retrieve(
    framework="pytorch",
    region=aws_region,
    version="2.6.0",
    py_version="py312",
    instance_type=instance_type,
    image_scope="training"
)


# Configure the local code uploaded to the training container.
source_code = SourceCode(
    source_dir=str(current_directory),
    entry_script="train.py",
    requirements="requirements.txt"
)


# Configure the remote SageMaker training instance.
compute = Compute(
    instance_type=instance_type,
    instance_count=1,
    volume_size_in_gb=30,
    enable_managed_spot_training=False
)


# Configure the maximum runtime of the training job.
stopping_condition = StoppingCondition(
    max_runtime_in_seconds=2 * 60 * 60
)


# Create the SageMaker V3 ModelTrainer.
estimator = ModelTrainer(
    training_image=training_image,
    source_code=source_code,
    role=execution_role,
    compute=compute,
    stopping_condition=stopping_condition,

    hyperparameters={
        "epochs": epochs,
        "batch-size": batch_size,
        "learning-rate": learning_rate
    },

    base_job_name="mushroom-resnet18"
)

print("ModelTrainer created.")
print("Training image:", training_image)
print("Source directory:", current_directory)
print("Instance type:", instance_type)
print("Epochs:", epochs)
print("Training has not started yet.")

[08/03/26 15:50:12] INFO     SageMaker session not provided. Using default Session.                  ]8;id=5169682;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=5169683;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Cannot simulate policies for                                  ]8;id=5169690;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=5169691;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::206440129399:role/service-role/AmazonSageMaker-                         
                             ExecutionRole-20260730T162226' (access denied); permission                            
                             verdict unknown.                                                                      

                    WARNING  Could not verify permissions for role                         ]8;id=5169697;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=5169698;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::206440129399:role/service-role/AmazonSageMaker-                         
                             ExecutionRole-20260730T162226' (caller lacks                                          
                             iam:SimulatePrincipalPolicy). Proceeding with it. If the                              
                             operation later fails with an access-denied error, ensure the                         
                             role has the required permissions for 'training' (see                                 
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Created S3 bucket: sagemaker-us-east-1-206440129399              ]8;id=5169705;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py\session_helper.py]8;;\:]8;id=5169706;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/session_helper.py#815\815]8;;\

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=5169712;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=5169713;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-east-1-206440129399/mushroom-resnet1                
                             8' kms_key_id=None compression_type='GZIP'                                            

                    INFO     Training image URI:                                               ]8;id=5169720;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=5169721;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.6                     
                             .0-cpu-py312                                                                          

ModelTrainer created.
Training image: 763104351884.dkr.ecr.us-east-1.amazonaws.com/pytorch-training:2.6.0-cpu-py312
Source directory: /home/sagemaker-user/ClassMushroomAWS/training
Instance type: ml.t3.large
Epochs: 1
Training has not started yet.


In [ ]:
# Define the SageMaker training channels.
training_inputs = {
    "train": TrainingInput(
        train_s3_uri,
        content_type="application/x-parquet"
    ),
    "validation": TrainingInput(
        validation_s3_uri,
        content_type="application/x-parquet"
    )
}

In [ ]:
# Launch the SageMaker Training Job.
estimator.fit(
    training_inputs,
    wait=False
)

print("Training job launched.")
print("Job name:", estimator.latest_training_job.name)

In [ ]:
wait=False